# HRNet hand-keypoint swap — PHW & KIM only (Kaggle)

Tests whether MediaPipe is the bottleneck for the two tracker-suspect signers PHW (0.898 CER, 0.716 visibility) and KIM (0.723 CER, 0.738 visibility), per the post-Task-A prompt.

**Single-variable test contract**: ONLY the keypoint backend changes.  The existing Stage 1 v2 (= Stage 1 v3 no_dann) model is evaluated unchanged.

**Three backends**:
1. `mediapipe_default`   — current behaviour (confidence 0.3)
2. `mediapipe_sensitive` — confidence dropped to 0.2 (recall-up control)
3. `rtmpose_hand`        — MMPose RTMPose-M hand5 model (HRNet-equivalent)

**Prerequisites**:
- `stage1v3_fold1_no_dann_best.pt` — KIM's fold checkpoint
- `stage1v3_fold2_no_dann_best.pt` — PHW's fold checkpoint
- `skeleton_features_t32.pt`       — MediaPipe-default cache (Cell 6 needs it ONLY if checkpoints must be rebuilt)
- `subject_cv5.json`               — CV manifest (always required for the fold splits)

Attach the most recent committed Stage 1 v3 / Task A kernel's output as a dataset to get these files.

**Wall-clock budget on Kaggle**:

| Path | Time |
|---|---|
| Checkpoints present     | ~50 min (3 extracts + eval) |
| Need to retrain fold 1+2 | ~2 h 30 min (T4 GPU) |

Use T4 if any retraining is needed; otherwise CPU is enough.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy imageio --quiet
# MMPose install for RTMPose backend.  Optional — script falls back to MediaPipe-only if missing.
!pip install -U openmim --quiet
!mim install 'mmengine' 'mmcv>=2.0.1' 'mmdet>=3.1.0' 'mmpose>=1.3.0' 2>&1 | tail -5 || echo 'mmpose install failed — script will skip the RTMPose backend.'

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')

## Cell 2 — Locate prerequisites

Auto-finds checkpoints, cache, and manifest under `/kaggle/input/` (any nesting) or `/kaggle/working/`.  Lists what's there so missing pieces are obvious.

In [ ]:
import os, glob, json

def _find(pattern: str):
    matches = (
        glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
        + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True)
    )
    return matches[0] if matches else None

CKPT_FOLD1 = _find('stage1v3_fold1_no_dann_best.pt')
CKPT_FOLD2 = _find('stage1v3_fold2_no_dann_best.pt')
CACHE_PATH = _find('skeleton_features_t32.pt')
CV_MANIFEST = _find('subject_cv5.json')
RESULTS_JSON = _find('stage1v3_results.json')

OUT_DIR_CACHES  = '/kaggle/working/caches/hrnet_swap'
OUT_DIR_REPORTS = '/kaggle/working/reports/hrnet_swap'
CKPT_DIR        = '/kaggle/working/checkpoints'
os.makedirs(OUT_DIR_CACHES,  exist_ok=True)
os.makedirs(OUT_DIR_REPORTS, exist_ok=True)
os.makedirs(CKPT_DIR,        exist_ok=True)

for label, path in [('ckpt_fold1', CKPT_FOLD1), ('ckpt_fold2', CKPT_FOLD2),
                     ('cache',      CACHE_PATH), ('cv_manifest', CV_MANIFEST),
                     ('results_json', RESULTS_JSON)]:
    print(f'{label:<14s}: {path or "<missing>"}')

NEED_RETRAIN = (CKPT_FOLD1 is None) or (CKPT_FOLD2 is None)
print(f'\nretrain folds 1+2? {NEED_RETRAIN}')

## Cell 3 — (Conditional) build cache + manifest, then retrain fold 1 + 2 no_dann if missing

Self-contained: if the skeleton cache and/or CV manifest are absent, this cell builds them from scratch (~30 min) before retraining.  If checkpoints are present, the whole cell is a no-op.

Wall-clock breakdown when nothing exists yet:
- Stream + extract skeleton cache: ~30 min
- Build CV manifest:                <1 min
- Retrain fold 1 + fold 2 (T4):     ~100 min
- **Total worst case: ~135 min**

In [ ]:
if NEED_RETRAIN:
    import random, numpy as np
    SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig
    cfg = Config(
        data=DataConfig(
            hf_repo_id='yewon816/WiTA', lang='english',
            max_zips=None, max_frames=64, seed=SEED,
        ),
        encoder=EncoderConfig(arch='siglip'),
        train=TrainConfig(
            num_epochs=80, batch_size=32, lr=5e-4, weight_decay=5e-2,
            grad_clip=1.0, num_workers=2, warmup_pct=0.05, seed=SEED,
            checkpoint_dir=CKPT_DIR,
        ),
    ).build()

    # ----- 1) Build the skeleton cache if absent.  ~30 min. -----
    if CACHE_PATH is None or not os.path.exists(CACHE_PATH):
        from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
        from wita_v2.datasets.skeleton_cache  import extract_skeleton_features
        print('Building skeleton cache from scratch...')
        samples = stream_and_index_with_subjects(cfg)
        print(f'Streamed {len(samples)} clips across '
              f'{len({s for _,_,s in samples})} subjects')
        CACHE_PATH = '/kaggle/working/skeleton_features_t32.pt'
        extract_skeleton_features(
            samples=samples, out_path=CACHE_PATH,
            T_native=32, dtype=torch.float16,
        )

    cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
    print(f'cache ready: {len(cache["feats"])} clips, '
          f'detect_rate={cache.get("frame_detect_rate", 0)*100:.1f}%')

    # ----- 2) Build the CV manifest if absent.  Sub-second. -----
    if CV_MANIFEST is None or not os.path.exists(CV_MANIFEST):
        from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest
        print('Building CV manifest from scratch...')
        fake_samples = [(b'', cache['labels'][i], cache['subjects'][i])
                        for i in range(len(cache['feats']))]
        manifest = build_cv5_manifest(fake_samples, n_folds=5, seed=SEED)
        CV_MANIFEST = '/kaggle/working/subject_cv5.json'
        save_cv5_manifest(manifest, CV_MANIFEST)
    else:
        from wita_v2.datasets.cv_splits import load_cv5_manifest
        manifest = load_cv5_manifest(CV_MANIFEST)
    print(f'manifest ready: {manifest["n_subjects_total"]} signers, '
          f'{manifest["n_folds"]} folds')

    # ----- 3) Retrain whichever no_dann fold(s) are missing.  ~50 min each. -----
    from wita_v2.training.dann_train       import train_one_fold
    from wita_v2.datasets.cv_splits        import fold_indices, build_fold_signer_map
    from wita_v2.datasets.skeleton_augment import LandmarkAugment

    aug = LandmarkAugment()    # Stage 1 v2 defaults
    folds_to_run = []
    if CKPT_FOLD1 is None: folds_to_run.append(1)
    if CKPT_FOLD2 is None: folds_to_run.append(2)
    print(f'Retraining folds: {folds_to_run}')

    for fold in folds_to_run:
        train_idx, val_idx = fold_indices(manifest, fold, cache['subjects'])
        signer_map = build_fold_signer_map(manifest, fold)
        result = train_one_fold(
            cache=cache, train_idx=train_idx, val_idx=val_idx,
            signer_to_local=signer_map, cfg=cfg,
            fold=fold, variant='no_dann', alpha=0.0, gamma_grl=10.0,
            num_epochs=80, batch_size=32, lr_peak=5e-4,
            weight_decay=5e-2, grad_clip=1.0, dropout=0.2,
            d_model=256, n_layers=4, n_heads=4, conv_kernel=15,
            upsample=2, warmup_pct=0.05, transform=aug,
            checkpoint_dir=CKPT_DIR,
        )
        if fold == 1: CKPT_FOLD1 = result['checkpoint_path']
        if fold == 2: CKPT_FOLD2 = result['checkpoint_path']
        print(f'Fold {fold} done: best val CER = {result["best_val_cer"]:.4f}')
else:
    print('Both checkpoints present — skipping retraining.')

## Cell 4 — Extract PHW + KIM keypoints with all 3 backends

Downloads only the 4 zips that belong to PHW and KIM (~95% network savings vs the full Stage 1 stream).  ~45 min CPU.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
!python /kaggle/working/wita_v2/scripts/extract_phw_kim_keypoints.py \
    --target-signers PHW KIM \
    --out-dir        {OUT_DIR_CACHES} \
    --device         {DEVICE}

## Cell 5 — Run the evaluation (Stage 1 v2 model, swapped feature source)

In [ ]:
assert CKPT_FOLD1 and os.path.exists(CKPT_FOLD1), f'Missing fold 1 ckpt: {CKPT_FOLD1}'
assert CKPT_FOLD2 and os.path.exists(CKPT_FOLD2), f'Missing fold 2 ckpt: {CKPT_FOLD2}'
!python /kaggle/working/wita_v2/scripts/eval_phw_kim_hrnet.py \
    --cache-dir         {OUT_DIR_CACHES} \
    --checkpoint-fold1  {CKPT_FOLD1} \
    --checkpoint-fold2  {CKPT_FOLD2} \
    --out-dir           {OUT_DIR_REPORTS} \
    --device            {DEVICE}

## Cell 6 — Build the 1-page report

Aggregates the eval CSVs + verdict.json into a Markdown report at `reports/hrnet_swap/report.md` for the dissertation appendix.

In [ ]:
import os, json, pandas as pd
from IPython.display import Markdown, display

rep_df  = pd.read_csv(os.path.join(OUT_DIR_REPORTS, 'per_signer_results.csv'))
clip_df = pd.read_csv(os.path.join(OUT_DIR_REPORTS, 'per_clip_cer.csv'))
with open(os.path.join(OUT_DIR_REPORTS, 'verdict.json')) as f:
    verdict = json.load(f)

# Build (backend × signer) table.
table = rep_df.pivot(index='backend', columns='signer', values='cer')
delta_table = pd.DataFrame({
    'ΔPHW vs default': verdict['baseline_phw'] - table['PHW'],
    'ΔKIM vs default': verdict['baseline_kim'] - table['KIM'],
})

lines = []
lines.append('# HRNet hand-keypoint swap — report\n')
lines.append('## Setup\n')
lines.append('Single-variable test: swap MediaPipe HandLandmarker for sensitive-MediaPipe / RTMPose ')
lines.append('on PHW + KIM clips ONLY; evaluate unchanged Stage 1 v2 (no_dann fold 2 / fold 1) checkpoint.')
lines.append('Everything else locked.\n')
lines.append('## Headline table\n')
lines.append(table.to_markdown(floatfmt='.4f') + '\n')
lines.append('## Δ vs mediapipe_default (positive = backend helps)\n')
lines.append(delta_table.to_markdown(floatfmt='+.4f') + '\n')
lines.append('## Verdict (per Task §4)\n')
cls = verdict.get('classification', 'unknown')
actions = {
    'strong_pass':  '**Commit to the winning backend full-dataset in Stage 3.** Re-extract all 3477 clips with the winner, rerun Stage 1 v2 5-fold CV.',
    'partial_pass': '**Worth swapping for the easier-to-fix signer; the harder one is a deeper filming problem.**  Document the asymmetry.',
    'null':         '**Accept PHW and KIM as a dataset-side limit.** Report all stages with full-cohort AND stripped-cohort numbers from now on.',
    'worse':        '**HRNet output distribution incompatible with the existing head.** Either briefly retrain on mixed cache or accept the dataset-side limit.',
}
lines.append(f'**Classification**: `{cls}`\n')
lines.append(actions.get(cls, '*(unknown classification)*') + '\n')
lines.append('## Per-clip scatter\n')
lines.append('![per-clip CER scatter](per_clip_scatter.png)\n')

report_md = '\n'.join(lines)
report_path = os.path.join(OUT_DIR_REPORTS, 'report.md')
with open(report_path, 'w') as f:
    f.write(report_md)
print(f'wrote {report_path}')
display(Markdown(report_md))

## Cell 7 — Commit kernel and pull artifacts

Save & Run All so:
- `reports/hrnet_swap/report.md`  → 1-page markdown for the dissertation appendix
- `reports/hrnet_swap/per_signer_results.csv`
- `reports/hrnet_swap/per_clip_cer.csv`
- `reports/hrnet_swap/per_clip_scatter.png`
- `reports/hrnet_swap/verdict.json`
- `caches/hrnet_swap/landmarks_*.pt` (so a follow-up Stage 3 kernel can attach them if needed)

all survive the session end.